# IOI: two-site cores in six models

Runs the GPU steps for each model, smallest first, and saves to `OUT` on Drive:

1. `golden`: sweep, scan and test, which finds the layers and ranks of the core
2. `wang`: the saved cores on the prompts of Wang et al. (2022)
3. `model_dump`: the per-prompt data for the reduced model
4. `controls`: PCA of the activations and a random subspace
5. `wang_controls`, GPT-2 models only: the core against other directions on the prompts of Wang et al.

Needs a G4 runtime (Qwen's weights are about 65 GB) and an `HF_TOKEN` secret. A finished step is skipped if the notebook is rerun after a disconnect.

The reduced model, figures and tables run afterwards from the saved results, with no GPU; see the README.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import json, os
OUT = '/content/drive/MyDrive/ace_ioi/results/ioi'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['PYTHONPATH'] = 'src'
!pip -q install "transformers==4.49.0" accelerate scikit-learn nltk
!rm -rf /content/cores && git clone -q https://github.com/joshseth/cores /content/cores
%cd /content/cores

In [ ]:
MODELS = {'gpt2':         dict(hf='gpt2', dt='float32', attn='', bs=128, gbs=32, ref='7:10', stride=1),
          'gpt2-medium':  dict(hf='gpt2-medium', dt='float32', attn='', bs=128, gbs=32, ref='11:20', stride=1),
          'gpt2-large':   dict(hf='gpt2-large', dt='float32', attn='', bs=96, gbs=16, ref='17:28', stride=1),
          'llama-3.1-8b': dict(hf='meta-llama/Llama-3.1-8B', dt='bfloat16', attn='', bs=64, gbs=8, ref='12:23', stride=1),
          'gemma-2-9b':   dict(hf='google/gemma-2-9b', dt='bfloat16', attn='--attn-impl eager', bs=48, gbs=8, ref='13:33', stride=1),
          'qwen2.5-32b':  dict(hf='Qwen/Qwen2.5-32B', dt='bfloat16', attn='', bs=48, gbs=4, ref='33:56', stride=2)}
RULE = 'flip at both sites'   # or 'flip at S2', 'swap at S2', 'swap at both sites'
QUIET = 'grep --line-buffered -v -i -E "warning|oneDNN|tensorflow|cudart"'
finished = lambda path: os.path.exists(path) and json.load(open(path)).get('finished')
for name, m in MODELS.items():
    g, d = f'{OUT}/golden/{name}', f'{OUT}/model/{name}'
    print(f'\n===================== {name}')
    !mkdir -p {g} {d}
    base = f"--model-name {m['hf']} --tag {name} --dtype {m['dt']} {m['attn']} --batch-size {m['bs']}"
    # 1. sweep, scan, test
    if not finished(f'{g}/{name}.json'):
        !python -m cores.ioi.golden {base} --reference {m['ref']} --stride {m['stride']} --grad-batch-size {m['gbs']} --out-dir {g} 2>&1 | {QUIET} | tee {g}/{name}.log
    # 2. the prompts of Wang et al.
    if not finished(f'{g}/{name}_wang.json'):
        !python -m cores.ioi.wang {base} --dir {g} 2>&1 | {QUIET} | tee {g}/{name}_wang.log
    # the layer pair with the smallest core; a tie goes to the lower removal on train
    r = json.load(open(f'{g}/{name}.json'))
    total = {q: sum(v['core'][RULE]['ranks']) for q, v in r['pairs'].items() if v['core'][RULE]}
    q = min(total, key=lambda k: (total[k], r['pairs'][k]['core'][RULE]['train']['remove']))
    lS, lE = r['pairs'][q]['layers']; rS, rE = r['pairs'][q]['core'][RULE]['ranks']
    print(f'{RULE}: layers S2 {lS}, END {lE}; core S2 {rS} + END {rE} = {rS + rE}', flush=True)
    # 3. per-prompt data for the reduced model
    if not os.path.exists(f'{d}/{name}_dump.npz'):
        !python -m cores.ioi.model_dump {base} --golden-dir {g} --layers {lS}:{lE} --ranks {rS}:{rE} --out {d}/{name}_dump.npz 2>&1 | {QUIET} | tail -3 | tee {d}/{name}_dump.log
    # 4. controls
    if not os.path.exists(f'{d}/{name}_controls.json'):
        !python -m cores.ioi.controls {base} --golden-dir {g} --layers {lS}:{lE} --ranks {rS}:{rE} --out {d}/{name}_controls.json 2>&1 | {QUIET} | tee {d}/{name}_controls.log
    # 5. the core against other directions on the prompts of Wang et al.
    if name.startswith('gpt2') and not os.path.exists(f'{OUT}/wang_controls_{name}.json'):
        !python -m cores.ioi.wang_controls {base} --golden-dir {g} --layers {lS}:{lE} --ranks {rS}:{rE} --out {OUT}/wang_controls_{name}.json 2>&1 | {QUIET} | tee {OUT}/wang_controls_{name}.log
    !rm -rf ~/.cache/huggingface/hub/models--* 2>/dev/null